In [ ]:
# In[1]:

import pandas as pd

# Load CSVs
metric = pd.read_csv('metric.csv')
trace = pd.read_csv('trace.csv')
log = pd.read_csv('log.csv')
error_logs = pd.read_csv('error_logs.csv')

# Parse timestamps as UTC datetimes (Unix seconds -> UTC)
metric['timestamp'] = pd.to_datetime(metric['timestamp'], unit='s', utc=True)
trace['timestamp'] = pd.to_datetime(trace['timestamp'], unit='s', utc=True)
log['timestamp'] = pd.to_datetime(log['timestamp'], unit='s', utc=True)
error_logs['timestamp'] = pd.to_datetime(error_logs['timestamp'], unit='s', utc=True)

# 1) metric.csv aggregates
cmdb_counts_metric = metric['cmdb_id'].value_counts().reset_index().rename(columns={'index':'cmdb_id','cmdb_id':'count'}).head(20)
kpi_counts_metric = metric['kpi_name'].value_counts().reset_index().rename(columns={'index':'kpi_name','kpi_name':'count'}).head(20)
metric_ts_range = pd.DataFrame({
    'min_timestamp_utc': [metric['timestamp'].min()],
    'max_timestamp_utc': [metric['timestamp'].max()]
})

# 2) trace.csv aggregates
cmdb_counts_trace = trace['cmdb_id'].value_counts().reset_index().rename(columns={'index':'cmdb_id','cmdb_id':'count'}).head(20)
trace_name_counts = trace['trace_name'].value_counts().reset_index().rename(columns={'index':'trace_name','trace_name':'count'}).head(20)
trace_ts_range = pd.DataFrame({
    'min_timestamp_utc': [trace['timestamp'].min()],
    'max_timestamp_utc': [trace['timestamp'].max()]
})

# 3) log.csv aggregates
cmdb_counts_log = log['cmdb_id'].value_counts().reset_index().rename(columns={'index':'cmdb_id','cmdb_id':'count'}).head(20)
log_name_counts = log['log_name'].value_counts().reset_index().rename(columns={'index':'log_name','log_name':'count'}).head(20)
log_ts_range = pd.DataFrame({
    'min_timestamp_utc': [log['timestamp'].min()],
    'max_timestamp_utc': [log['timestamp'].max()]
})

# 4) error_logs.csv aggregates
total_error_rows = len(error_logs)
cmdb_counts_error = error_logs['cmdb_id'].value_counts().reset_index().rename(columns={'index':'cmdb_id','cmdb_id':'count'}).head(20)
sample_error_rows = error_logs.sort_values('timestamp')[['timestamp','cmdb_id','message']].head(5)

# Display compact results (variables separated by commas)
cmdb_counts_metric, kpi_counts_metric, metric_ts_range, \
cmdb_counts_trace, trace_name_counts, trace_ts_range, \
cmdb_counts_log, log_name_counts, log_ts_range, \
total_error_rows, cmdb_counts_error, sample_error_rows

```
Out[1]:
```
summary = (
    "Summary of telemetry files (UTC):\n\n"
    "metric.csv:\n"
    "- Top components by row count: orders (200), front-end (175), carts (153), "
    "catalogue (150), user (150), shipping (150), payment (150), orders-db (100), etc.\n"
    "- Top KPI names: cpu (375), mem (375), socket (375), diskio (200), latency-90 (175), "
    "latency-50 (175), workload (175), error (53).\n"
    "- Time range: 2024-01-20 08:58:00+00:00 to 2024-01-20 09:22:00+00:00.\n\n"
    "trace.csv:\n"
    "- No trace data present (empty).\n\n"
    "log.csv:\n"
    "- Top components by row count: catalogue (50), front-end (50), orders (50), user (50), "
    "payment (50), queue-master (50), shipping (50), carts-db (48), carts (48), session-db (10), user-db (4).\n"
    "- Log fields: log.error_count (230), log.row_count (230).\n"
    "- Time range: 2024-01-20 08:58:00+00:00 to 2024-01-20 09:22:00+00:00.\n\n"
    "error_logs.csv:\n"
    "- Total rows: 0 (no error log samples available).\n\n"
    "Overall: metric and log telemetry are available for the 08:58–09:22 UTC window; trace and error_logs are empty."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(                count  count
0              orders    200
1           front-end    175
2               carts    153
3           catalogue    150
4                user    150
5            shipping    150
6             payment    150
7           orders-db    100
8        catalogue-db    100
9            carts-db    100
10       queue-master    100
11         session-db    100
12           rabbitmq    100
13            user-db    100
14  rabbitmq-exporter     75,         count  count
0         cpu    375
1         mem    375
2      socket    375
3      diskio    200
4  latency-90    175
5  latency-50    175
6    workload    175
7       error     53,           min_timestamp_utc         max_timestamp_utc
0 2024-01-20 08:58:00+00:00 2024-01-20 09:22:00+00:00, Empty DataFrame
Columns: [count, count]
Index: [], Empty DataFrame
Columns: [count, count]
Index: [],   min_timestamp_utc max_timestamp_utc
0               NaT               NaT,            count  count
0      catalogue     50
1      front-end     50
2         orders     50
3           user     50
4        payment     50
5   queue-master     50
6       shipping     50
7       carts-db     48
8          carts     48
9     session-db     10
10       user-db      4,              count  count
0  log.error_count    230
1    log.row_count    230,           min_timestamp_utc         max_timestamp_utc
0 2024-01-20 08:58:00+00:00 2024-01-20 09:22:00+00:00, 0, Empty DataFrame
Columns: [count, count]
Index: [], Empty DataFrame
Columns: [timestamp, cmdb_id, message]
Index: [])```
```

In [ ]:
# In[2]:

import numpy as np
import pandas as pd

# Reuse existing metric and log DataFrames from previous steps.
# Ensure timestamp columns are parsed as UTC datetimes (idempotent if already done).
metric['timestamp'] = pd.to_datetime(metric['timestamp'], unit='s', utc=True)
log['timestamp'] = pd.to_datetime(log['timestamp'], unit='s', utc=True)

# Incident window (UTC)
start_ts = pd.to_datetime("2024-01-20 08:55:30", utc=True)
end_ts   = pd.to_datetime("2024-01-20 09:25:30", utc=True)

# --- 1) metric.csv analysis ---

# Compute global P95 per (cmdb_id, kpi_name) using all available rows (rule 9)
p95_metric = metric.groupby(['cmdb_id', 'kpi_name'], as_index=False)['value'].quantile(0.95)
p95_metric = p95_metric.rename(columns={'value': 'p95_threshold'})

# Filter metric rows to incident window
metric_window = metric[(metric['timestamp'] >= start_ts) & (metric['timestamp'] <= end_ts)].copy()

# Merge windowed rows with global P95 thresholds
metric_merged = metric_window.merge(p95_metric, on=['cmdb_id', 'kpi_name'], how='left')

# Mark breaches (value >= p95)
metric_merged['is_breach'] = metric_merged['value'] >= metric_merged['p95_threshold']

# Aggregations per (cmdb_id, kpi_name)
metric_rows = metric_merged.groupby(['cmdb_id', 'kpi_name'], as_index=False).agg(
    rows_in_window=('value', 'size'),
    breach_count=('is_breach', 'sum'),
    max_value=('value', 'max'),
    p95_threshold=('p95_threshold', 'first'),
)

# earliest breach timestamp (where is_breach True)
first_breach_series = (
    metric_merged[metric_merged['is_breach']]
    .groupby(['cmdb_id', 'kpi_name'])['timestamp']
    .min()
    .rename('first_breach_utc')
    .reset_index()
)

metric_rows = metric_rows.merge(first_breach_series, on=['cmdb_id', 'kpi_name'], how='left')

# ratio max_value / p95_threshold (handle zero or NaN thresholds)
metric_rows['ratio'] = np.where(
    metric_rows['p95_threshold'] > 0,
    metric_rows['max_value'] / metric_rows['p95_threshold'],
    np.nan
)

# Keep only required columns and sort by breach_count desc, then rows_in_window desc
metric_result = metric_rows[
    ['cmdb_id', 'kpi_name', 'rows_in_window', 'breach_count', 'first_breach_utc', 'max_value', 'p95_threshold', 'ratio']
].sort_values(['breach_count', 'rows_in_window'], ascending=[False, False])

# If no breaches at all, report empty DataFrame per instructions
if metric_result['breach_count'].sum() == 0:
    metric_top20 = metric_result.iloc[0:0]  # empty with same columns
else:
    metric_top20 = metric_result[metric_result['breach_count'] > 0].head(20)

# --- 2) log.csv analysis ---

# Compute global P95 per (cmdb_id, log_name) using all available rows
p95_log = log.groupby(['cmdb_id', 'log_name'], as_index=False)['value'].quantile(0.95)
p95_log = p95_log.rename(columns={'value': 'p95_threshold'})

# Filter log rows to incident window
log_window = log[(log['timestamp'] >= start_ts) & (log['timestamp'] <= end_ts)].copy()

# Merge windowed rows with global P95 thresholds
log_merged = log_window.merge(p95_log, on=['cmdb_id', 'log_name'], how='left')
log_merged['is_breach'] = log_merged['value'] >= log_merged['p95_threshold']

# Aggregations per (cmdb_id, log_name)
log_rows = log_merged.groupby(['cmdb_id', 'log_name'], as_index=False).agg(
    rows_in_window=('value', 'size'),
    breach_count=('is_breach', 'sum'),
    max_value=('value', 'max'),
    p95_threshold=('p95_threshold', 'first'),
)

# earliest breach timestamp (where is_breach True)
first_breach_log = (
    log_merged[log_merged['is_breach']]
    .groupby(['cmdb_id', 'log_name'])['timestamp']
    .min()
    .rename('first_breach_utc')
    .reset_index()
)

log_rows = log_rows.merge(first_breach_log, on=['cmdb_id', 'log_name'], how='left')

# ratio
log_rows['ratio'] = np.where(
    log_rows['p95_threshold'] > 0,
    log_rows['max_value'] / log_rows['p95_threshold'],
    np.nan
)

log_result = log_rows[
    ['cmdb_id', 'log_name', 'rows_in_window', 'breach_count', 'first_breach_utc', 'max_value', 'p95_threshold', 'ratio']
].sort_values(['breach_count', 'rows_in_window'], ascending=[False, False])

if log_result['breach_count'].sum() == 0:
    log_top20 = log_result.iloc[0:0]
else:
    log_top20 = log_result[log_result['breach_count'] > 0].head(20)

# Display compact results (top 20 each or empty)
metric_top20, log_top20

```
Out[2]:
```
Summary (incident window 2024-01-20 08:55:30 — 2024-01-20 09:25:30 UTC):

Overall finding
- Many metric KPIs and several log metrics exceeded their global P95 during the window. First breaches mostly start at 2024-01-20 08:58:00 UTC.

Metric highlights (top breaches)
- Repeated full-window breaches (breach_count = 25) for socket-related KPIs across services: catalogue, catalogue-db (also diskio), orders-db, payment, queue-master, rabbitmq, session-db. First breach time: 2024-01-20 08:58:00+00:00.
- Several of those have max_value == p95_threshold (ratio ≈ 1.0), indicating values at or equal to the P95 (stable high).
- Some KPIs have p95_threshold == 0 (e.g., catalogue-db diskio, rabbitmq diskio, front-end error, orders error, etc.), producing NaN for ratio; these show breaches counted but threshold is zero so ratio is undefined.
- Carts shows multiple KPI breaches but smaller counts: cpu, latency-50, latency-90, mem, socket each had 2 breaches in the window. Ratios are slightly >1 for these (e.g., carts cpu ratio ≈ 2.46; carts mem ratio ≈ 1.0007), with first breaches around 09:05–09:21 UTC.
- user-db socket had 4 breaches (first at 08:58).

Log highlights (top breaches)
- Many components recorded log.error_count breaches for every row in window (breach_count = rows_in_window, often 25). However their p95_threshold and max_value are 0 in many cases, producing NaN ratio (threshold==0).
- log.row_count shows meaningful breaches for some services:
  - session-db: 5 rows, 5 breaches (max=5, p95=5.0, ratio=1.0).
  - payment, queue-master, shipping, catalogue, front-end, orders, user, carts, carts-db: some have breach_count 2–3 with ratios >1 (notably carts log.row_count ratio ≈ 1.92, carts-db ratio ≈ 4.86), first breaches around 09:02–09:16 UTC.

Interpretation and notes
- Socket-related KPIs across many services are repeatedly at or above their global P95 at the start of the window (08:58 UTC), suggesting a widespread socket-level signal to investigate.
- Several diskio/error metrics have p95 == 0; treat those carefully (thresholds of zero make ratio meaningless). For those, breach counts indicate nonzero flagging behavior relative to zero baseline and should be inspected (could be telemetry normalization/zero-baseline issue).
- Carts and some log.row_count entries show notable spikes above P95 (ratios >1), worth inspecting for increased load or log churn.
- Next steps: inspect raw traces/log messages for services with high socket breaches (catalogue, orders-db, rabbitmq, payment, queue-master, session-db) and validate telemetry normalization for metrics with p95==0.

The original code execution output of IPython Kernel is also provided below for reference:

(              cmdb_id    kpi_name  rows_in_window  breach_count          first_breach_utc     max_value  p95_threshold     ratio
15          catalogue      socket              25            25 2024-01-20 08:58:00+00:00  7.000000e+00   7.000000e+00  1.000000
18       catalogue-db      diskio              25            25 2024-01-20 08:58:00+00:00  0.000000e+00   0.000000e+00       NaN
20       catalogue-db      socket              25            25 2024-01-20 08:58:00+00:00  4.000000e+00   4.000000e+00  1.000000
22          front-end       error              25            25 2024-01-20 08:58:00+00:00  0.000000e+00   0.000000e+00       NaN
30             orders       error              25            25 2024-01-20 08:58:00+00:00  0.000000e+00   0.000000e+00       NaN
39          orders-db      socket              25            25 2024-01-20 08:58:00+00:00  9.000000e+00   9.000000e+00  1.000000
44            payment      socket              25            25 2024-01-20 08:58:00+00:00  6.000000e+00   6.000000e+00  1.000000
49       queue-master      socket              25            25 2024-01-20 08:58:00+00:00  3.000000e+00   3.000000e+00  1.000000
51           rabbitmq      diskio              25            25 2024-01-20 08:58:00+00:00  0.000000e+00   0.000000e+00       NaN
53           rabbitmq      socket              25            25 2024-01-20 08:58:00+00:00  1.100000e+01   1.100000e+01  1.000000
55  rabbitmq-exporter         mem              25            25 2024-01-20 08:58:00+00:00  4.730880e+06   4.730880e+06  1.000000
56  rabbitmq-exporter      socket              25            25 2024-01-20 08:58:00+00:00  1.000000e+00   1.000000e+00  1.000000
60         session-db      socket              25            25 2024-01-20 08:58:00+00:00  3.000000e+00   3.000000e+00  1.000000
76            user-db      socket              25             4 2024-01-20 08:58:00+00:00  1.100000e+01   1.100000e+01  1.000000
1               carts       error               3             3 2024-01-20 09:20:00+00:00  0.000000e+00   0.000000e+00       NaN
0               carts         cpu              25             2 2024-01-20 09:05:00+00:00  7.008616e+00   2.850520e+00  2.458715
2               carts  latency-50              25             2 2024-01-20 09:20:00+00:00  2.263904e-02   2.224625e-02  1.017657
3               carts  latency-90              25             2 2024-01-20 09:18:00+00:00  4.727323e-02   4.673939e-02  1.011422
4               carts         mem              25             2 2024-01-20 09:21:00+00:00  2.154431e+08   2.152832e+08  1.000743
5               carts      socket              25             2 2024-01-20 09:05:00+00:00  1.251667e+01   1.213333e+01  1.031593,          cmdb_id         log_name  rows_in_window  breach_count          first_breach_utc  max_value  p95_threshold     ratio
4      catalogue  log.error_count              25            25 2024-01-20 08:58:00+00:00          0           0.00       NaN
6      front-end  log.error_count              25            25 2024-01-20 08:58:00+00:00          0           0.00       NaN
8         orders  log.error_count              25            25 2024-01-20 08:58:00+00:00          0           0.00       NaN
10       payment  log.error_count              25            25 2024-01-20 08:58:00+00:00          0           0.00       NaN
12  queue-master  log.error_count              25            25 2024-01-20 08:58:00+00:00          0           0.00       NaN
16      shipping  log.error_count              25            25 2024-01-20 08:58:00+00:00          0           0.00       NaN
18          user  log.error_count              25            25 2024-01-20 08:58:00+00:00          0           0.00       NaN
0          carts  log.error_count              24            24 2024-01-20 08:59:00+00:00          0           0.00       NaN
2       carts-db  log.error_count              24            24 2024-01-20 08:58:00+00:00          0           0.00       NaN
14    session-db  log.error_count               5             5 2024-01-20 08:59:00+00:00          0           0.00       NaN
15    session-db    log.row_count               5             5 2024-01-20 08:59:00+00:00          5           5.00  1.000000
11       payment    log.row_count              25             3 2024-01-20 09:02:00+00:00        108         105.00  1.028571
13  queue-master    log.row_count              25             3 2024-01-20 09:02:00+00:00        612         585.00  1.046154
17      shipping    log.row_count              25             3 2024-01-20 09:02:00+00:00         68          65.00  1.046154
5      catalogue    log.row_count              25             2 2024-01-20 09:04:00+00:00        173         166.80  1.037170
7      front-end    log.row_count              25             2 2024-01-20 09:02:00+00:00       2064        2013.00  1.025335
9         orders    log.row_count              25             2 2024-01-20 09:02:00+00:00        136         129.80  1.047766
19          user    log.row_count              25             2 2024-01-20 09:02:00+00:00        629         608.40  1.033859
1          carts    log.row_count              24             2 2024-01-20 09:16:00+00:00        226         117.50  1.923404
3       carts-db    log.row_count              24             2 2024-01-20 08:59:00+00:00          9           1.85  4.864865)```
```

In [ ]:
# In[3]:

import numpy as np
import pandas as pd

# Reuse metric and log DataFrames; ensure timestamps are UTC datetimes
metric['timestamp'] = pd.to_datetime(metric['timestamp'], unit='s', utc=True)
log['timestamp'] = pd.to_datetime(log['timestamp'], unit='s', utc=True)

# Incident window (UTC)
start_ts = pd.to_datetime("2024-01-20 08:55:30", utc=True)
end_ts   = pd.to_datetime("2024-01-20 09:25:30", utc=True)

# Helper to detect consecutive-minute runs among breach rows (assumes rows sorted by timestamp)
def detect_runs(df_breach):
    if df_breach.empty:
        return {
            'total_breach_count': 0,
            'number_of_runs': 0,
            'longest_run_length': 0,
            'longest_run_start_utc': pd.NaT,
            'longest_run_end_utc': pd.NaT,
            'longest_run_duration_minutes': 0,
            'max_value_in_longest_run': np.nan,
        }
    ts = df_breach['timestamp'].sort_values().reset_index(drop=True)
    diffs = ts.diff().dt.total_seconds().fillna(9999)
    new_run = diffs != 60  # new run when gap != 60s
    run_id = new_run.cumsum()
    runs = pd.DataFrame({'ts': ts, 'run_id': run_id})
    run_stats = runs.groupby('run_id').agg(
        run_start=('ts', 'first'),
        run_end=('ts', 'last'),
        run_len=('ts', 'size')
    ).reset_index(drop=True)
    # attach max value per run by merging timestamps back to values
    df_vals = df_breach.set_index('timestamp')
    run_stats['max_value'] = run_stats.apply(
        lambda r: df_vals.loc[r['run_start']:r['run_end'],'value'].max(), axis=1
    )
    # choose longest run (if tie, pick earliest)
    longest_idx = run_stats['run_len'].idxmax()
    longest = run_stats.loc[longest_idx] if not run_stats.empty else None
    return {
        'total_breach_count': len(df_breach),
        'number_of_runs': len(run_stats),
        'longest_run_length': int(longest['run_len']) if longest is not None else 0,
        'longest_run_start_utc': longest['run_start'] if longest is not None else pd.NaT,
        'longest_run_end_utc': longest['run_end'] if longest is not None else pd.NaT,
        'longest_run_duration_minutes': int(longest['run_len']) if longest is not None else 0,
        'max_value_in_longest_run': float(longest['max_value']) if longest is not None else np.nan,
    }

# --- 1) Socket metrics fault-run detection ---

# Compute global P95 per cmdb_id for kpi_name == 'socket' across full metric series
socket_all = metric[metric['kpi_name'] == 'socket'].copy()
p95_socket = socket_all.groupby('cmdb_id', as_index=False)['value'].quantile(0.95).rename(columns={'value':'p95_threshold'})

# Filter socket rows to incident window
socket_window = socket_all[(socket_all['timestamp'] >= start_ts) & (socket_all['timestamp'] <= end_ts)].copy()

# For each cmdb_id present in p95_socket, detect breaches and runs
socket_rows = []
for _, row in p95_socket.iterrows():
    cmdb = row['cmdb_id']
    p95 = row['p95_threshold']
    df_win = socket_window[socket_window['cmdb_id'] == cmdb].sort_values('timestamp')
    # breaches in window (value >= p95)
    if pd.isna(p95):
        # If p95 is NaN (no data), treat as no breaches
        df_breach = df_win.iloc[0:0]
    else:
        df_breach = df_win[df_win['value'] >= p95].copy()
    runs = detect_runs(df_breach)
    # compute ratio (handle p95 == 0)
    ratio = np.nan
    if (p95 is not None) and (not pd.isna(p95)) and (p95 != 0):
        ratio = runs['max_value_in_longest_run'] / p95 if runs['max_value_in_longest_run'] == runs['max_value_in_longest_run'] else np.nan
    # append record if meets criteria (total_breach_count >=5 OR at least one run length >=2)
    if (runs['total_breach_count'] >= 5) or (runs['longest_run_length'] >= 2):
        socket_rows.append({
            'cmdb_id': cmdb,
            'total_breach_count': runs['total_breach_count'],
            'number_of_runs': runs['number_of_runs'],
            'longest_run_length': runs['longest_run_length'],
            'longest_run_start_utc': runs['longest_run_start_utc'],
            'longest_run_end_utc': runs['longest_run_end_utc'],
            'longest_run_duration_minutes': runs['longest_run_duration_minutes'],
            'max_value_in_longest_run': runs['max_value_in_longest_run'],
            'p95_threshold': 0.0 if (not pd.isna(p95) and p95 == 0) else p95,
            'ratio': np.nan if (not pd.isna(p95) and p95 == 0) else ratio
        })

socket_results = pd.DataFrame(socket_rows)
if not socket_results.empty:
    socket_results = socket_results.sort_values(['total_breach_count','longest_run_length'], ascending=[False, False]).head(20)
else:
    socket_results = socket_results.reindex(columns=[
        'cmdb_id','total_breach_count','number_of_runs','longest_run_length',
        'longest_run_start_utc','longest_run_end_utc','longest_run_duration_minutes',
        'max_value_in_longest_run','p95_threshold','ratio'
    ])

# --- 2) Carts CPU and log.row_count detection ---

# carts cpu (metric)
p95_carts_cpu = metric[(metric['cmdb_id']=='carts') & (metric['kpi_name']=='cpu')]['value'].quantile(0.95)
carts_cpu_win = metric[(metric['cmdb_id']=='carts') & (metric['kpi_name']=='cpu') & (metric['timestamp']>=start_ts) & (metric['timestamp']<=end_ts)].sort_values('timestamp')
carts_cpu_breach = carts_cpu_win[carts_cpu_win['value'] >= p95_carts_cpu].copy()
carts_cpu_runs = detect_runs(carts_cpu_breach)
carts_cpu_ratio = np.nan
if (not pd.isna(p95_carts_cpu)) and (p95_carts_cpu != 0):
    carts_cpu_ratio = carts_cpu_runs['max_value_in_longest_run'] / p95_carts_cpu if carts_cpu_runs['max_value_in_longest_run'] == carts_cpu_runs['max_value_in_longest_run'] else np.nan

carts_cpu_result = pd.DataFrame([{
    'cmdb_id': 'carts',
    'kpi_name': 'cpu',
    'total_breach_count': carts_cpu_runs['total_breach_count'],
    'number_of_runs': carts_cpu_runs['number_of_runs'],
    'longest_run_start_utc': carts_cpu_runs['longest_run_start_utc'],
    'longest_run_end_utc': carts_cpu_runs['longest_run_end_utc'],
    'longest_run_duration_minutes': carts_cpu_runs['longest_run_duration_minutes'],
    'max_value_in_longest_run': carts_cpu_runs['max_value_in_longest_run'],
    'p95_threshold': 0.0 if (not pd.isna(p95_carts_cpu) and p95_carts_cpu == 0) else p95_carts_cpu,
    'ratio': np.nan if (not pd.isna(p95_carts_cpu) and p95_carts_cpu == 0) else carts_cpu_ratio
}])

# carts and carts-db log.row_count
log_pairs = [('carts','log.row_count'), ('carts-db','log.row_count')]
log_rows_out = []
for cmdb, lname in log_pairs:
    p95_logpair = log[(log['cmdb_id']==cmdb) & (log['log_name']==lname)]['value'].quantile(0.95)
    log_win = log[(log['cmdb_id']==cmdb) & (log['log_name']==lname) & (log['timestamp']>=start_ts) & (log['timestamp']<=end_ts)].sort_values('timestamp')
    log_breach = log_win[log_win['value'] >= p95_logpair].copy()
    log_runs = detect_runs(log_breach)
    ratio = np.nan
    if (not pd.isna(p95_logpair)) and (p95_logpair != 0):
        ratio = log_runs['max_value_in_longest_run'] / p95_logpair if log_runs['max_value_in_longest_run'] == log_runs['max_value_in_longest_run'] else np.nan
    log_rows_out.append({
        'cmdb_id': cmdb,
        'kpi_name': lname,
        'total_breach_count': log_runs['total_breach_count'],
        'number_of_runs': log_runs['number_of_runs'],
        'longest_run_start_utc': log_runs['longest_run_start_utc'],
        'longest_run_end_utc': log_runs['longest_run_end_utc'],
        'longest_run_duration_minutes': log_runs['longest_run_duration_minutes'],
        'max_value_in_longest_run': log_runs['max_value_in_longest_run'],
        'p95_threshold': 0.0 if (not pd.isna(p95_logpair) and p95_logpair == 0) else p95_logpair,
        'ratio': np.nan if (not pd.isna(p95_logpair) and p95_logpair == 0) else ratio
    })

carts_log_results = pd.DataFrame(log_rows_out)

# Display compact results (variables)
socket_results, carts_cpu_result, carts_log_results

```
Out[3]:
```
Summary of fault-run detection (incident window 2024-01-20 08:55:30 — 2024-01-20 09:25:30 UTC)

1) Socket metric runs (cmdb_id-level)
- Widespread, continuous socket breaches across the window (each with total_breach_count = 25, longest_run_length = 25 spanning 2024-01-20 08:58:00 → 2024-01-20 09:22:00 UTC):
  - catalogue — breaches 25, 1 run, longest 25 min, max_in_longest = 7.0, p95 = 7.0, ratio = 1.00
  - catalogue-db — breaches 25, 1 run, longest 25 min, max = 4.0, p95 = 4.0, ratio = 1.00
  - orders-db — breaches 25, 1 run, longest 25 min, max = 9.0, p95 = 9.0, ratio = 1.00
  - payment — breaches 25, 1 run, longest 25 min, max = 6.0, p95 = 6.0, ratio = 1.00
  - queue-master — breaches 25, 1 run, longest 25 min, max = 3.0, p95 = 3.0, ratio = 1.00
  - rabbitmq — breaches 25, 1 run, longest 25 min, max = 11.0, p95 = 11.0, ratio = 1.00
  - rabbitmq-exporter — breaches 25, 1 run, longest 25 min, max = 1.0, p95 = 1.0, ratio = 1.00
  - session-db — breaches 25, 1 run, longest 25 min, max = 3.0, p95 = 3.0, ratio = 1.00
- Additional shorter/partial socket signals:
  - user-db — breaches 4, 1 run, longest 4 min (08:58 → 09:01), max = 11.0, p95 = 11.0, ratio = 1.00
  - carts-db — breaches 2, 1 run, longest 2 min (08:58 → 08:59), max = 7.0, p95 ≈ 6.546667, ratio ≈ 1.069
  - user — breaches 2, 1 run, longest 2 min (08:58 → 08:59), max = 23.0, p95 ≈ 22.466667, ratio ≈ 1.024

Interpretation: many services show a full-window, continuous socket-level breach (values equal to their historical P95 for the service) starting at 08:58 UTC. Ratios ≈1.0 indicate values at or very near P95.

2) Carts CPU and log.row_count runs

- carts — metric cpu:
  - total_breach_count = 2, number_of_runs = 2, longest run length = 1 (single-minute runs)
  - longest run at 2024-01-20 09:05:00 UTC (duration 1 min), max_in_longest = 2.933434, p95 ≈ 2.850520, ratio ≈ 1.029

- carts — log.row_count:
  - total_breach_count = 2, number_of_runs = 2
  - longest run at 2024-01-20 09:16:00 UTC (1 min), max_in_longest = 125.0, p95 = 117.50, ratio ≈ 1.064

- carts-db — log.row_count:
  - total_breach_count = 2, number_of_runs = 2
  - longest run at 2024-01-20 08:59:00 UTC (1 min), max_in_longest = 2.0, p95 = 1.85, ratio ≈ 1.081

Overall takeaway (compact)
- The dominant signal is a synchronous, continuous socket-level elevation across many services (catalogue, catalogue-db, orders-db, payment, queue-master, rabbitmq, rabbitmq-exporter, session-db) starting ~08:58 UTC and lasting the entire observed window — these are primary candidates for investigation.
- Carts shows small, short CPU and log.row_count breaches (2 isolated minutes each) worth checking but much less sustained than the socket-wide runs.
- Focus next on socket-level diagnostics (network/socket connectivity, service endpoints, and any upstream changes) for the listed services.

The original code execution output of IPython Kernel is also provided below for reference:

(              cmdb_id  total_breach_count  number_of_runs  longest_run_length     longest_run_start_utc       longest_run_end_utc  longest_run_duration_minutes  max_value_in_longest_run  p95_threshold     ratio
1           catalogue                  25               1                  25 2024-01-20 08:58:00+00:00 2024-01-20 09:22:00+00:00                            25                       7.0       7.000000  1.000000
2        catalogue-db                  25               1                  25 2024-01-20 08:58:00+00:00 2024-01-20 09:22:00+00:00                            25                       4.0       4.000000  1.000000
3           orders-db                  25               1                  25 2024-01-20 08:58:00+00:00 2024-01-20 09:22:00+00:00                            25                       9.0       9.000000  1.000000
4             payment                  25               1                  25 2024-01-20 08:58:00+00:00 2024-01-20 09:22:00+00:00                            25                       6.0       6.000000  1.000000
5        queue-master                  25               1                  25 2024-01-20 08:58:00+00:00 2024-01-20 09:22:00+00:00                            25                       3.0       3.000000  1.000000
6            rabbitmq                  25               1                  25 2024-01-20 08:58:00+00:00 2024-01-20 09:22:00+00:00                            25                      11.0      11.000000  1.000000
7   rabbitmq-exporter                  25               1                  25 2024-01-20 08:58:00+00:00 2024-01-20 09:22:00+00:00                            25                       1.0       1.000000  1.000000
8          session-db                  25               1                  25 2024-01-20 08:58:00+00:00 2024-01-20 09:22:00+00:00                            25                       3.0       3.000000  1.000000
10            user-db                   4               1                   4 2024-01-20 08:58:00+00:00 2024-01-20 09:01:00+00:00                             4                      11.0      11.000000  1.000000
0            carts-db                   2               1                   2 2024-01-20 08:58:00+00:00 2024-01-20 08:59:00+00:00                             2                       7.0       6.546667  1.069246
9                user                   2               1                   2 2024-01-20 08:58:00+00:00 2024-01-20 08:59:00+00:00                             2                      23.0      22.466667  1.023739,   cmdb_id kpi_name  total_breach_count  number_of_runs     longest_run_start_utc       longest_run_end_utc  longest_run_duration_minutes  max_value_in_longest_run  p95_threshold     ratio
0   carts      cpu                   2               2 2024-01-20 09:05:00+00:00 2024-01-20 09:05:00+00:00                             1                  2.933434        2.85052  1.029087,     cmdb_id       kpi_name  total_breach_count  number_of_runs     longest_run_start_utc       longest_run_end_utc  longest_run_duration_minutes  max_value_in_longest_run  p95_threshold     ratio
0     carts  log.row_count                   2               2 2024-01-20 09:16:00+00:00 2024-01-20 09:16:00+00:00                             1                     125.0         117.50  1.063830
1  carts-db  log.row_count                   2               2 2024-01-20 08:59:00+00:00 2024-01-20 08:59:00+00:00                             1                       2.0           1.85  1.081081)```
```